# RoBERTa fine-tuning for phishing detection
**Full run:** In Google Colab choose Runtime -> Change runtime type -> **GPU**, then Runtime -> **Run all**.
With a GPU, the notebook trains on the full sample and downloads `roberta_results.csv` plus `roberta-phishing-model.zip`.
If no GPU is available, it falls back to a small CPU demo run. That still creates the model archive, but skips evaluation; do not use that demo model for project metrics or real decisions.


In [ ]:
!pip install -q transformers scikit-learn pandas
!rm -rf /content/Phishing && git clone -q https://github.com/krishitha11/Phishing.git /content/Phishing
!ls /content/Phishing/data

In [ ]:
DATA_DIR = '/content/Phishing/data'
MODEL_NAME = 'roberta-base'
MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 2
LR = 2e-5
MAX_TRAIN_ROWS = None  # GPU default: use all 20,000 rows
SEED = 42


In [ ]:
import torch, random, numpy as np, pandas as pd
from contextlib import nullcontext
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)
if device == 'cpu':
    # Short CPU-only demo so a checkpoint can still be created without a GPU.
    MAX_LEN = 128
    BATCH_SIZE = 2
    EPOCHS = 1
    MAX_TRAIN_ROWS = 128
    print('CPU demo mode: 128 training emails, 1 epoch, no evaluation. This checkpoint is for interface demonstration only.')
else:
    MAX_TRAIN_ROWS = None

train = pd.read_csv(f'{DATA_DIR}/train.csv')
if MAX_TRAIN_ROWS:
    train = train.sample(min(MAX_TRAIN_ROWS, len(train)), random_state=SEED)
val = pd.read_csv(f'{DATA_DIR}/val.csv')
val_limit = 3000 if device == 'cuda' else 1
val = val.sample(min(val_limit, len(val)), random_state=SEED)
train['text'] = train['text'].astype(str); val['text'] = val['text'].astype(str)
print(len(train), 'training emails,', len(val), 'validation emails')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

In [ ]:
def predict(texts, batch_size=128):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok(list(texts[i:i+batch_size]), truncation=True, max_length=MAX_LEN,
                      padding=True, return_tensors="pt").to(device)
            amp_context = torch.autocast('cuda', dtype=torch.float16) if device == 'cuda' else nullcontext()
            with amp_context:
                logits = model(**enc).logits
            preds.extend(logits.argmax(-1).cpu().tolist())
    return np.array(preds)

In [ ]:
texts = train['text'].tolist(); labels = train['label'].tolist()
steps = max(1, (len(texts) // BATCH_SIZE) * EPOCHS)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

for epoch in range(EPOCHS):
    model.train()
    order = np.random.permutation(len(texts))
    running = 0.0
    for step, i in enumerate(range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE)):
        idx = order[i:i + BATCH_SIZE]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(device)
        y = torch.tensor([labels[j] for j in idx]).to(device)
        amp_context = torch.autocast('cuda', dtype=torch.float16) if device == 'cuda' else nullcontext()
        with amp_context:
            loss = model(**enc, labels=y).loss
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()
        if (step + 1) % 100 == 0:
            print(f"epoch {epoch+1}  step {step+1}  loss {running/100:.4f}")
            running = 0.0
    if device == 'cuda':
        val_acc = (predict(val['text'].tolist()) == val['label'].values).mean()
        print(f'=== epoch {epoch+1} done. Validation accuracy: {val_acc:.4f}')
    else:
        print(f'=== epoch {epoch+1} done. CPU demo; validation skipped.')

In [ ]:
from pathlib import Path
import shutil

MODEL_EXPORT = Path('/content/roberta-phishing-model')
model.save_pretrained(MODEL_EXPORT)
tok.save_pretrained(MODEL_EXPORT)
archive_path = shutil.make_archive('/content/roberta-phishing-model', 'zip', root_dir=MODEL_EXPORT)
print('Saved checkpoint archive:', archive_path)


In [ ]:
if device == 'cuda':
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

    def score(name, df):
        pred = predict(df['text'].astype(str).tolist())
        y = df['label'].values
        p, r, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        return dict(test_set=name, accuracy=accuracy_score(y, pred), precision=p, recall=r, f1=f1,
                    false_positive_rate=fp / (fp + tn) if (fp + tn) else 0.0, rows=len(df))

    files = {'clean': 'test.csv', 'typos': 'test_typos.csv', 'synonyms': 'test_synonyms.csv',
             'char_swaps': 'test_char_swaps.csv', 'all_combined': 'test_all_combined.csv',
             'external': 'test_external.csv'}
    rows = []
    for name, f in files.items():
        rows.append(score(name, pd.read_csv(f'{DATA_DIR}/{f}')))
        print('done', name)
    res = pd.DataFrame(rows)
    res.to_csv(f'{DATA_DIR}/roberta_results.csv', index=False)
    print(res.round(4).to_string(index=False))
else:
    res = None
    print('Skipping evaluation in CPU demo mode; no metrics CSV was created.')

In [ ]:
from google.colab import files
if res is not None:
    res.to_csv('/content/roberta_results.csv', index=False)
    files.download('/content/roberta_results.csv')
files.download(archive_path)
print('Downloaded the model archive. GPU runs also download the results CSV.')